## Utility
#### This Notebook is made to check the resulted curated files of all data sources after running the orchedstrator in training mode 

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import rasterio

### GADM Curated check

In [2]:
communes = gpd.read_file("../data/training/curated/boundaries/algeria_communes.gpkg")
wilayas  = gpd.read_file("../data/training/curated/boundaries/algeria_wilayas.gpkg")

# Shape

print(f"\n1. SHAPE")

print(f"   Wilayas:  {len(wilayas)}  (expected: 48)")
print(f"   Communes: {len(communes)} (expected: ~1541-1560)")

print(f"   Wilayas  {'OK' if len(wilayas) == 48 else 'ERROR'}")
print(f"   Communes {'OK' if 1500 <= len(communes) <= 1600 else 'ERROR'}")

# CRS

print(f"\n2. CRS")

print(f"   Wilayas:  {wilayas.crs}  {'OK' if wilayas.crs.to_epsg() == 4326 else 'ERROR'}")
print(f"   Communes: {communes.crs} {'OK' if communes.crs.to_epsg() == 4326 else 'ERROR'}")

# Geometry validity

invalid_w = (~wilayas.geometry.is_valid).sum()
invalid_c = (~communes.geometry.is_valid).sum()

print(f"\n3. GEOMETRY VALIDITY")

print(f"   Invalid wilayas:  {invalid_w}  {'OK' if invalid_w == 0 else 'ERROR'}")
print(f"   Invalid communes: {invalid_c} {'OK' if invalid_c == 0 else 'ERROR'}")

# Key columns

print(f"\n4. KEY COLUMNS")

for col in ["GID_1", "GID_2", "NAME_1", "NAME_2"]:
    present = col in communes.columns
    nulls   = communes[col].isna().sum() if present else "N/A"
    print(f"   {col}: {'OK' if present else 'MISSING'}  nulls={nulls}")

# GID_2 uniqueness (join key must be unique)

dupes = communes["GID_2"].duplicated().sum()

print(f"\n5. GID_2 UNIQUENESS (join key)")

print(f"   Duplicates: {dupes} {'OK' if dupes == 0 else 'DUPLICATES FOUND'}")

# Bbox sanity

b = communes.total_bounds

print(f"\n6. BBOX (Algeria: roughly -8.7W 18.9N 12.0E 37.1N)")
print(f"   minx={b[0]:.2f} miny={b[1]:.2f} maxx={b[2]:.2f} maxy={b[3]:.2f}")

in_range = (-9 < b[0] < -8) and (18 < b[1] < 20) and (11 < b[2] < 13) and (36 < b[3] < 38)

print(f"   {'OK' if in_range else 'OUT OF RANGE'}")


1. SHAPE
   Wilayas:  48  (expected: 48)
   Communes: 1504 (expected: ~1541-1560)
   Wilayas  OK
   Communes OK

2. CRS
   Wilayas:  EPSG:4326  OK
   Communes: EPSG:4326 OK

3. GEOMETRY VALIDITY
   Invalid wilayas:  0  OK
   Invalid communes: 0 OK

4. KEY COLUMNS
   GID_1: OK  nulls=0
   GID_2: OK  nulls=0
   NAME_1: OK  nulls=0
   NAME_2: OK  nulls=0

5. GID_2 UNIQUENESS (join key)
   Duplicates: 0 OK

6. BBOX (Algeria: roughly -8.7W 18.9N 12.0E 37.1N)
   minx=-8.67 miny=18.96 maxx=11.99 maxy=37.09
   OK


### FIRMS Curated check

In [3]:
firms = pd.read_parquet("../data/training/curated/firms/firms_curated.parquet")

print("FIRMS VALIDATION")
print(f"{'='*55}")

# Shape
print(f"\n1. SHAPE")
print(f"   Rows: {len(firms):,}  (expect > 10,000 for multi-year)")
print(f"   {'PASS' if len(firms) > 10_000 else 'FAIL — TOO FEW ROWS'}")

# Date range
firms["acq_date"] = pd.to_datetime(firms["acq_date"])

print(f"\n2. DATE RANGE")
print(f"   Min: {firms['acq_date'].min().date()}")
print(f"   Max: {firms['acq_date'].max().date()}")
print(f"   Years covered: {sorted(firms['acq_date'].dt.year.unique())}")

# Fire season concentration
month_counts = firms["acq_date"].dt.month.value_counts().sort_index()

peak = month_counts.loc[[6, 7, 8, 9]].sum()
total = month_counts.sum()

print(f"\n3. FIRE SEASON CONCENTRATION (Jun-Sep should dominate)")

for m, c in month_counts.items():
    bar = "█" * (c // max(1, total // 50))
    print(f"   Month {m:02d}: {c:>6,} {bar}")

# Commune join coverage
print(f"\n4. COMMUNE JOIN (GID_2)")

has_gid2 = "GID_2" in firms.columns

print(
    f"   GID_2 column present: "
    f"{'PASS' if has_gid2 else 'FAIL — MISSING — rerun firms.py curate()'}"
)

if has_gid2:
    null_gid = firms["GID_2"].isna().sum()

    print(
        f"   Null GID_2: {null_gid:,} "
        f"({100 * null_gid / len(firms):.1f}%) "
        f"{'PASS' if null_gid / len(firms) < 0.05 else 'FAIL — TOO MANY'}"
    )

    print(f"   Unique communes with fires: {firms['GID_2'].nunique()}")

# FRP sanity
print(f"\n5. FRP SANITY")
print(f"   Min FRP:  {firms['frp'].min():.1f} MW")
print(f"   Max FRP:  {firms['frp'].max():.1f} MW")
print(f"   Mean FRP: {firms['frp'].mean():.1f} MW")

negatives = (firms["frp"] < 0).sum()

print(
    f"   Negative FRP: {negatives} "
    f"{'PASS' if negatives == 0 else 'FAIL'}"
)

# Confidence values
print(f"\n6. CONFIDENCE VALUES (should be 66 or 99 only)")
print(f"   {firms['confidence'].value_counts().to_dict()}")

valid_conf = set(firms["confidence"].unique()) <= {66, 99}

print(
    f"   {'PASS — Only 66/99' if valid_conf else 'FAIL — unexpected values'}"
)

# Gas flare zones — should have near-zero detections
print(f"\n7. GAS FLARE CHECK (should be minimal detections near oilfields)")

flare_check = firms[
    (firms["latitude"].between(31.4, 32.0)) &
    (firms["longitude"].between(5.8, 6.4))
]

print(
    f"   Detections near Hassi Messaoud: {len(flare_check)} "
    f"{'PASS' if len(flare_check) < 50 else 'FAIL — POSSIBLE FLARE CONTAMINATION'}"
)

FIRMS VALIDATION

1. SHAPE
   Rows: 134,974  (expect > 10,000 for multi-year)
   PASS

2. DATE RANGE
   Min: 2015-05-01
   Max: 2025-11-01
   Years covered: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

3. FIRE SEASON CONCENTRATION (Jun-Sep should dominate)
   Month 05:  8,224 ███
   Month 06: 11,492 ████
   Month 07: 32,550 ████████████
   Month 08: 49,172 ██████████████████
   Month 09: 16,121 █████
   Month 10: 17,058 ██████
   Month 11:    357 

4. COMMUNE JOIN (GID_2)
   GID_2 column present: PASS
   Null GID_2: 0 (0.0%) PASS
   Unique communes with fires: 1356

5. FRP SANITY
   Min FRP:  0.0 MW
   Max FRP:  1855.4 MW
   Mean FRP: 13.0 MW
   Negative FRP: 0 PASS

6. CONFIDENCE VALUES (should be 66 or 99 only)
   {66: 121325, 99: 13649}
   PASS — Only 66/99

7. GAS FLARE CHECK (should be minimal detections near oilfields)
   Detections near Hassi Messa

### ERA5 Curated Check

In [4]:
era5 = pd.read_parquet(sorted(Path("../data/training/curated/era5").glob("*.parquet"))[-1])

era5["date"] = pd.to_datetime(era5["date"])

print("ERA5 VALIDATION")

print(f"{'='*55}")

# Shape

print(f"\n1. SHAPE")

print(f"   Rows:       {len(era5):,}")

print(f"   ERA5 cells: {era5['era5_cell_id'].nunique()}")

print(f"   Dates:      {era5['date'].nunique()}")

print(f"   {'PASS' if len(era5) > 1000 else 'TOO FEW'}")

# Date range

print(f"\n2. DATE RANGE")

print(f"   Min: {era5['date'].min().date()}")

print(f"   Max: {era5['date'].max().date()}")

months = era5["date"].dt.month.unique()

print(f"   Months present: {sorted(months)} (fire season only — expect Jun-Sep)")

# Duplicates

dupes = era5.duplicated(subset=["date", "era5_cell_id"]).sum()

print(f"\n3. DUPLICATES (date × era5_cell_id)")

print(f"   Duplicates: {dupes} {'PASS' if dupes == 0 else 'RERUN CURATE'}")

# NaN check

print(f"\n4. MISSING VALUES")

for col in [
    "temp_c", "rh", "wind_speed_kmh", "precip_mm",
    "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"
]:
    n = era5[col].isna().sum()
    print(f"   {col:<16}: {n:,} {'PASS' if n == 0 else 'MISSING VALUES'}")

# Physical range checks

print(f"\n5. PHYSICAL RANGE CHECKS")

checks = [
    ("temp_c",        -10,  55,  era5["temp_c"].min(),         era5["temp_c"].max()),
    ("rh",               0, 100,  era5["rh"].min(),             era5["rh"].max()),
    ("wind_speed_kmh",   0, 150,  era5["wind_speed_kmh"].min(), era5["wind_speed_kmh"].max()),
    ("precip_mm",        0, 200,  era5["precip_mm"].min(),      era5["precip_mm"].max()),
    ("FWI",              0, 300,  era5["FWI"].min(),             era5["FWI"].max()),
    ("FFMC",             0, 101,  era5["FFMC"].min(),            era5["FFMC"].max()),
]

for col, lo, hi, vmin, vmax in checks:
    ok = (vmin >= lo) and (vmax <= hi)

    print(
        f"   {col:<18}: min={vmin:>7.2f}  max={vmax:>7.2f} "
        f"(expected {lo}–{hi}) {'PASS' if ok else 'OUT OF RANGE'}"
    )

# FWI fire season stats

print(f"\n6. FWI FIRE SEASON STATS")

peak = era5[era5["date"].dt.month.isin([7, 8])]

print(f"   Jul-Aug mean FWI: {peak['FWI'].mean():.1f} (expect > 15 for Algeria)")

print(f"   Jul-Aug max FWI:  {peak['FWI'].max():.1f}")

print(
    f"   {'PASS' if peak['FWI'].mean() > 10 else 'FWI TOO LOW — check precip accumulation bug'}"
)

# Precipitation accumulation bug check

print(f"\n7. PRECIPITATION SANITY")

print(
    f"   Days with precip > 100mm: "
    f"{(era5['precip_mm'] > 100).sum()} "
    f"(should be very few in fire season)"
)

print(f"   Mean precip (fire season): {era5['precip_mm'].mean():.2f}mm/day")

status = "PASS" if era5["precip_mm"].mean() < 5 else "HIGH — possible accumulation bug"

print(f"   {status}")

ERA5 VALIDATION

1. SHAPE
   Rows:       43,010,000
   ERA5 cells: 21250
   Dates:      2024
   PASS

2. DATE RANGE
   Min: 2015-05-01
   Max: 2025-10-31
   Months present: [np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)] (fire season only — expect Jun-Sep)

3. DUPLICATES (date × era5_cell_id)
   Duplicates: 0 PASS

4. MISSING VALUES
   temp_c          : 0 PASS
   rh              : 0 PASS
   wind_speed_kmh  : 0 PASS
   precip_mm       : 0 PASS
   FFMC            : 0 PASS
   DMC             : 0 PASS
   DC              : 0 PASS
   ISI             : 0 PASS
   BUI             : 0 PASS
   FWI             : 0 PASS

5. PHYSICAL RANGE CHECKS
   temp_c            : min=   3.89  max=  48.10 (expected -10–55) PASS
   rh                : min=   2.23  max= 100.00 (expected 0–100) PASS
   wind_speed_kmh    : min=   0.00  max=  65.01 (expected 0–150) PASS
   precip_mm         : min=   0.00  max= 156.56 (expected 0–200) PASS
   FWI               : min=   0.00  max= 238.6

### Sentinel Curated Check

In [5]:
sentinel = pd.read_parquet(
    sorted(Path("../data/training/curated/sentinel").glob("*.parquet"))[-1]
)

sentinel["date"] = pd.to_datetime(sentinel["date"])

print("SENTINEL-2 VALIDATION")

print(f"{'='*55}")

print(f"\n1. SHAPE")

print(f"   Rows:     {len(sentinel):,}")

print(f"   Communes: {sentinel['commune_id'].nunique()}")

print(f"   Months:   {sentinel['date'].nunique()}")

print(f"\n2. DATE RANGE")

print(f"   Min: {sentinel['date'].min().date()}")

print(f"   Max: {sentinel['date'].max().date()}")

print(f"\n3. MISSING VALUES (after forward-fill — should be 0)")

for col in ["NDVI", "NDWI", "NBR"]:
    n = sentinel[col].isna().sum()
    print(f"   {col}: {n:,} {'PASS' if n == 0 else 'MISSING VALUES'}")

print(f"\n4. VALUE RANGE (all indices must be -1 to 1)")

for col in ["NDVI", "NDWI", "NBR"]:
    vmin, vmax = sentinel[col].min(), sentinel[col].max()

    ok = vmin >= -1 and vmax <= 1

    print(
        f"   {col}: min={vmin:.3f}  max={vmax:.3f}  "
        f"{'PASS' if ok else 'OUT OF RANGE'}"
    )

print(f"\n5. NDVI SEASONAL PATTERN (should drop Jul-Aug — dry season)")

monthly = sentinel.groupby(sentinel["date"].dt.month)["NDVI"].mean()

for m, v in monthly.items():
    bar = "█" * int(v * 30)
    print(f"   Month {m:02d}: {v:.3f} {bar}")

peak_month = monthly.idxmax()
trough_month = monthly.idxmin()

print(f"   Peak month: {peak_month}  Trough: {trough_month}")

print(
    f"   {'PASS' if trough_month in [7, 8, 9] else 'Trough not in Jul-Sep — check'}"
)

print(f"\n6. COMMUNE COVERAGE")

total_communes = 1504

covered = sentinel["commune_id"].nunique()

print(
    f"   Communes with Sentinel data: {covered} / {total_communes} "
    f"({100 * covered / total_communes:.1f}%)"
)

print(
    f"   {'PASS' if covered / total_communes > 0.85 else 'LOW COVERAGE'}"
)

SENTINEL-2 VALIDATION

1. SHAPE
   Rows:     96,256
   Communes: 1504
   Months:   64

2. DATE RANGE
   Min: 2015-07-01
   Max: 2025-10-01

3. MISSING VALUES (after forward-fill — should be 0)
   NDVI: 0 PASS
   NDWI: 0 PASS
   NBR: 0 PASS

4. VALUE RANGE (all indices must be -1 to 1)
   NDVI: min=-0.477  max=0.823  PASS
   NDWI: min=-0.303  max=0.945  PASS
   NBR: min=-0.256  max=0.943  PASS

5. NDVI SEASONAL PATTERN (should drop Jul-Aug — dry season)
   Month 05: 0.314 █████████
   Month 06: 0.249 ███████
   Month 07: 0.216 ██████
   Month 08: 0.207 ██████
   Month 09: 0.221 ██████
   Month 10: 0.242 ███████
   Peak month: 5  Trough: 8
   PASS

6. COMMUNE COVERAGE
   Communes with Sentinel data: 1504 / 1504 (100.0%)
   PASS


### Landcover Curated Check

In [6]:
lc = pd.read_parquet("../data/training/curated/landcover/landcover_communes.parquet")

print("LANDCOVER VALIDATION")

print(f"{'='*55}")

print(f"\n1. SHAPE")

print(
    f"   Rows: {len(lc)} (expected: ~1504) "
    f"{'PASS' if 1500 <= len(lc) <= 1560 else 'INVALID'}"
)

print(f"\n2. ALL-ZERO COMMUNES")

all_zero = lc[
    [
        "forest_fraction",
        "shrub_fraction",
        "grass_fraction",
        "crop_fraction",
        "urban_fraction",
        "bare_fraction",
    ]
].sum(axis=1) == 0

print(
    f"   Count: {all_zero.sum()} "
    f"{'PASS' if all_zero.sum() == 0 else 'INVALID'}"
)

print(f"\n3. KNOWN FIRE-PRONE WILAYAS (burnable > 0.05)")

check_wilayas = {
    "Béjaïa":      "DZA.8_1",
    "Jijel":       "DZA.23_1",
    "Skikda":      "DZA.39_1",
    "Tizi Ouzou":  "DZA.47_1",
    "Tlemcen":     "DZA.48_1",
    "Annaba":      "DZA.5_1",
    "El Tarf":     "DZA.19_1",
    "Chlef":       "DZA.14_1",
}

for name, gid in check_wilayas.items():
    subset = lc[lc["wilaya_id"] == gid]

    if len(subset) == 0:
        print(f"   {name}: NOT FOUND")
        continue

    b = subset["burnable_fraction"].mean()

    print(
        f"   {'PASS' if b > 0.05 else 'INVALID'} "
        f"{name}: burnable={b:.3f} ({len(subset)} communes)"
    )

print(f"\n4. SAHARAN WILAYAS (burnable < threshold)")

sahara_check = {
    "Adrar":       ("DZA.1_1",  0.02),
    "Tamanrasset": ("DZA.41_1", 0.50),
    "Tindouf":     ("DZA.44_1", 0.05),
    "Illizi":      ("DZA.22_1", 0.05),
    "El Oued":     ("DZA.18_1", 0.05),
}

for name, (gid, thresh) in sahara_check.items():
    subset = lc[lc["wilaya_id"] == gid]

    b = subset["burnable_fraction"].mean()

    print(
        f"   {'PASS' if b < thresh else 'INVALID'} "
        f"{name}: burnable={b:.4f} (threshold <{thresh})"
    )

LANDCOVER VALIDATION

1. SHAPE
   Rows: 1504 (expected: ~1504) PASS

2. ALL-ZERO COMMUNES
   Count: 0 PASS

3. KNOWN FIRE-PRONE WILAYAS (burnable > 0.05)
   PASS Béjaïa: burnable=0.874 (48 communes)
   PASS Jijel: burnable=0.897 (28 communes)
   PASS Skikda: burnable=0.808 (38 communes)
   PASS Tizi Ouzou: burnable=0.859 (67 communes)
   PASS Tlemcen: burnable=0.655 (53 communes)
   PASS Annaba: burnable=0.614 (12 communes)
   PASS El Tarf: burnable=0.706 (24 communes)
   PASS Chlef: burnable=0.610 (35 communes)

4. SAHARAN WILAYAS (burnable < threshold)
   PASS Adrar: burnable=0.0021 (threshold <0.02)
   PASS Tamanrasset: burnable=0.0007 (threshold <0.5)
   PASS Tindouf: burnable=0.0001 (threshold <0.05)
   PASS Illizi: burnable=0.0007 (threshold <0.05)
   PASS El Oued: burnable=0.0360 (threshold <0.05)


### DEM Curated Check

In [12]:
dem_files = {
    "elevation": Path("../data/training/curated/dem/elevation.tif"),
    "slope":     Path("../data/training/curated/dem/slope.tif"),
    "aspect":    Path("../data/training/curated/dem/aspect.tif"),
}

for name, path in dem_files.items():
    print(f"\n── {name.upper()} ──")
    if not path.exists():
        print(f"   FILE NOT FOUND: {path}")
        continue
    with rasterio.open(path) as src:
        print(
            f"   CRS:        {src.crs} "
            f"{'PASS' if src.crs.to_epsg() == 4326 else 'NOT EPSG:4326'}"
        )
        print(f"   Resolution: {src.res}")
        print(f"   Shape:      {src.height} × {src.width}")
        print(f"   Bounds:     {src.bounds}")
        # Check bounds cover Algeria
        b = src.bounds
        covers = b.left < -8 and b.right > 11 and b.bottom < 19 and b.top > 36
        print(f"   Covers Algeria: {'PASS' if covers else 'FAIL'}")
        # Read sample to check values
        data = src.read(1, masked=True)
        valid = data.compressed()
        if len(valid) == 0:
            print("   ALL NODATA — file is empty or corrupt")
            continue
        print(f"   Nodata %:   {100 * (data.mask.sum() / data.size):.1f}%")
        print(f"   Min:        {valid.min():.1f}")
        print(f"   Max:        {valid.max():.1f}")
        print(f"   Mean:       {valid.mean():.1f}")
        # Range checks per layer
        if name == "elevation":
            ok = valid.min() >= -50 and valid.max() <= 3100
            print(
                f"   Range check (-50 to 3100m): "
                f"{'PASS' if ok else 'FAIL'}"
            )
            # Ahaggar peak is ~2918m, Djurdjura ~2308m
            print(
                f"   {'PASS' if valid.max() > 2000 else 'MAX TOO LOW — missing mountain coverage'}"
            )
        elif name == "slope":
            ok = valid.min() >= 0 and valid.max() <= 90
            print(
                f"   Range check (0–90°): "
                f"{'PASS' if ok else 'FAIL'}"
            )
        elif name == "aspect":
            ok = valid.min() >= -1 and valid.max() <= 360
            print(
                f"   Range check (-1 to 360°): "
                f"{'PASS' if ok else 'FAIL'}"
            )


── ELEVATION ──
   CRS:        EPSG:4326 PASS
   Resolution: (0.009010569268207584, 0.009010172458250514)
   Shape:      2012 × 2293
   Bounds:     BoundingBox(left=-8.673868178999953, bottom=18.96023082700009, right=11.987367153000037, top=37.088697813000124)
   Covers Algeria: PASS
   Nodata %:   43.3%
   Min:        -33.0
   Max:        2640.8
   Mean:       558.8
   Range check (-50 to 3100m): PASS
   PASS

── SLOPE ──
   CRS:        EPSG:4326 PASS
   Resolution: (0.009010569268207584, 0.009010172458250514)
   Shape:      2012 × 2293
   Bounds:     BoundingBox(left=-8.673868178999953, bottom=18.96023082700009, right=11.987367153000037, top=37.088697813000124)
   Covers Algeria: PASS
   Nodata %:   43.3%
   Min:        0.0
   Max:        52.1
   Mean:       0.7
   Range check (0–90°): PASS

── ASPECT ──
   CRS:        EPSG:4326 PASS
   Resolution: (0.009010569268207584, 0.009010172458250514)
   Shape:      2012 × 2293
   Bounds:     BoundingBox(left=-8.673868178999953, bottom=18.96

### WORLDPOP Curated Check 

In [11]:
wp_path = Path("../data/training/curated/worldpop/population_density.tif")
if not wp_path.exists():
    print(f"FILE NOT FOUND: {wp_path}")
else:
    with rasterio.open(wp_path) as src:
        print(f"\n1. METADATA")
        print(
            f"   CRS:        {src.crs} "
            f"{'PASS' if src.crs.to_epsg() == 4326 else 'FAIL'}"
        )
        print(f"   Resolution: {src.res}")
        print(f"   Shape:      {src.height} × {src.width}")
        print(f"   Bands:      {src.count}")
        b = src.bounds
        covers = b.left < -8 and b.right > 11 and b.bottom < 19 and b.top > 36
        print(f"   Covers Algeria: {'PASS' if covers else 'FAIL'}")
        data = src.read(1, masked=True)
        valid = data.compressed()
        print(f"\n2. VALUE SANITY")
        print(f"   Nodata %:   {100 * (data.mask.sum() / data.size):.1f}%")
        print(f"   Min:        {valid.min():.2f} people/km²")
        print(f"   Max:        {valid.max():.2f} people/km²")
        print(f"   Mean:       {valid.mean():.2f} people/km²")
        print(f"   Median:     {np.median(valid):.2f} people/km²")
        # Algeria overall density ~18 people/km², Algiers district much higher
        negatives = (valid < 0).sum()
        print(f"\n3. RANGE CHECKS")
        print(
            f"   Negative values: {negatives} "
            f"{'PASS' if negatives == 0 else 'FAIL'}"
        )
        print(
            f"   Max < 50,000:    "
            f"{'PASS' if valid.max() < 50_000 else 'UNREALISTIC'}"
        )
        print(
            f"   Mean > 0.1:      "
            f"{'PASS' if valid.mean() > 0.1 else 'TOO LOW'}"
        )
        # High density check — Algiers/Oran should show up
        high_density = (valid > 1000).sum()
        print(
            f"   Cells > 1000/km²: {high_density} "
            f"{'PASS' if high_density > 0 else 'NO URBAN AREAS DETECTED'}"
        )


1. METADATA
   CRS:        EPSG:4326 PASS
   Resolution: (0.009010569268207584, 0.009010172458250514)
   Shape:      2012 × 2293
   Bands:      1
   Covers Algeria: PASS

2. VALUE SANITY
   Nodata %:   43.3%
   Min:        0.01 people/km²
   Max:        42417.37 people/km²
   Mean:       20.49 people/km²
   Median:     0.22 people/km²

3. RANGE CHECKS
   Negative values: 0 PASS
   Max < 50,000:    PASS
   Mean > 0.1:      PASS
   Cells > 1000/km²: 9639 PASS


### OSM Curated Check

In [10]:
road_path = Path("../data/training/curated/roads/road_distance.tif")
if not road_path.exists():
    print(f"FILE NOT FOUND: {road_path}")
else:
    with rasterio.open(road_path) as src:
        print(f"\n1. METADATA")
        print(
            f"   CRS:        {src.crs} "
            f"{'PASS' if src.crs.to_epsg() == 4326 else 'FAIL'}"
        )
        print(f"   Resolution: {src.res}")
        print(f"   Shape:      {src.height} × {src.width}")
        b = src.bounds
        covers = b.left < -8 and b.right > 11 and b.bottom < 19 and b.top > 36
        print(f"   Covers Algeria: {'PASS' if covers else 'FAIL'}")
        data = src.read(1, masked=True)
        valid = data.compressed()
        print(f"\n2. VALUE SANITY (distance in km)")
        print(f"   Nodata %:   {100 * (data.mask.sum() / data.size):.1f}%")
        print(f"   Min:        {valid.min():.3f} km  (should be ~0 near roads)")
        print(f"   Max:        {valid.max():.1f} km")
        print(f"   Mean:       {valid.mean():.2f} km")
        print(f"   Median:     {np.median(valid):.2f} km")
        print(f"\n3. RANGE CHECKS")
        negatives = (valid < 0).sum()
        print(
            f"   Negative values:     {negatives} "
            f"{'PASS' if negatives == 0 else 'FAIL'}"
        )
        # Max road distance in Algeria Sahara could be ~300km
        print(
            f"   Max < 1000km:         "
            f"{'PASS' if valid.max() < 1000 else 'UNREALISTIC'}"
        )
        # Near-zero values confirm roads are present
        near_road = (valid < 0.1).sum()
        print(
            f"   Cells within 100m of road: {near_road:,} "
            f"{'PASS' if near_road > 0 else 'NO NEAR-ROAD CELLS'}"
        )
        print(f"\n4. SPOT CHECK vs KNOWN ROADS")
        # Algiers city center should have ~0 road distance
        # Sample the pixel at roughly Algiers coords (36.7°N, 3.05°E)
        try:
            row, col = src.index(3.05, 36.7)
            algiers_val = src.read(1)[row, col]
            print(
                f"   Algiers center (~3.05E, 36.7N): "
                f"{algiers_val:.3f} km "
                f"{'PASS' if algiers_val < 1.0 else 'HIGHER THAN EXPECTED'}"
            )
        except Exception as e:
            print(f"   Spot check failed: {e}")
        # Deep Sahara should have large distance
        # (3.0, 25.0) : "Central Sahara"
        try:
            row, col = src.index(3.0, 25.0)
            sahara_val = src.read(1)[row, col]
            print(
                f"   Central Sahara (~3.0E, 25.0N):    "
                f"{sahara_val:.1f} km "
                f"{'PASS' if sahara_val > 10 else 'LOWER THAN EXPECTED'}"
            )
        except Exception as e:
            print(f"   Spot check failed: {e}")


1. METADATA
   CRS:        EPSG:4326 PASS
   Resolution: (0.009010569268207584, 0.009010172458250514)
   Shape:      2012 × 2293
   Covers Algeria: PASS

2. VALUE SANITY (distance in km)
   Nodata %:   0.0%
   Min:        0.000 km  (should be ~0 near roads)
   Max:        833.6 km
   Mean:       137.81 km
   Median:     66.48 km

3. RANGE CHECKS
   Negative values:     0 PASS
   Max < 1000km:         PASS
   Cells within 100m of road: 105,458 PASS

4. SPOT CHECK vs KNOWN ROADS
   Algiers center (~3.05E, 36.7N): 0.000 km PASS
   Central Sahara (~3.0E, 25.0N):    65.0 km PASS
